In [ ]:
from __future__ import annotations

import os
import re
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ============================================================
# User configuration
# ============================================================

root_dir = Path(os.getcwd())

dataset_names = [
    "Replogle_K562_essential",
    "Replogle_RPE",
    "NormanWeissman2019",
    "ChangYe",
    "ZhaoSims2021", 
]

groups = ["single", "dual", "multi"]

In [ ]:
# If True, process every existing dataset/group under root_dir.
# If False, only process SELECTED_SUB_PATH.
RUN_ALL_VALID_PATHS = True
SELECTED_SUB_PATH: Path | None = None

# Whether to keep S0 if it is selected in selected_variants_TEMPLATE_EDIT_ME.csv.
# Usually S0 is a reference rather than a selected strategy bar.
INCLUDE_S0_AS_BAR = False

# S0 is shown as a dashed horizontal reference line, not as a bar.
NAIVE_BASELINE_ID = "S0_naive_mean_control_reference"
SHOW_NAIVE_BASELINE_LINE = True
NAIVE_BASELINE_LINE_COLOR = "#EE5862"
NAIVE_BASELINE_LINESTYLE = "--"
NAIVE_BASELINE_LINEWIDTH = 1.15
NAIVE_BASELINE_LINE_ALPHA = 0.85
NAIVE_BASELINE_TEXT = "Naive average control"
NAIVE_BASELINE_TEXT_SIZE = 10
NAIVE_BASELINE_TEXT_COLOR = "#555555"
NAIVE_BASELINE_TEXT_X_FRACTION = 0.985
NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION = 0.012

# For local mixing score, 0.5 is the random-mixing / balanced-neighborhood reference.
SHOW_LOCAL_MIXING_HALF_REFERENCE = True
LOCAL_MIXING_HALF_REFERENCE_VALUE = 0.5
LOCAL_MIXING_HALF_REFERENCE_COLOR = "#555555"
LOCAL_MIXING_HALF_REFERENCE_LINESTYLE = ":"
LOCAL_MIXING_HALF_REFERENCE_LINEWIDTH = 1.55
LOCAL_MIXING_HALF_REFERENCE_ALPHA = 1.0
LOCAL_MIXING_HALF_REFERENCE_TEXT = "0.5 random-mixing reference"
LOCAL_MIXING_HALF_REFERENCE_TEXT_SIZE = 10
LOCAL_MIXING_HALF_REFERENCE_TEXT_COLOR = "#555555"

# Significance is tested against random single control only, excluding S0.
REFERENCE_STRATEGY_FOR_TEST = "S1_random_single_control"
REFERENCE_STRATEGY_FOR_TEST_LABEL = "Random single control"
REFERENCE_BAR_TEXT = "ref"
SHOW_REFERENCE_BAR_TEXT = False

# Use STRATEGY_PLOT_LABELS for bar x-axis labels.
# Variant suffixes such as (200&5) are appended on a second line when present,
# so multiple selected variants from the same strategy remain distinguishable.
USE_STRATEGY_PLOT_LABELS_FOR_XTICKS = True
ADD_VARIANT_SUFFIX_TO_XTICKS = True

# Statistics: "auto", "paired", or "unpaired".
# auto uses paired tests when every selected variant has the same seed IDs.
STAT_MODE = "paired"
ALPHA = 0.05

# Multiple-comparison correction used for pairwise tests.
# Pairwise p-values are Holm-adjusted.
PAIRWISE_CORRECTION = "holm"

# Plot appearance
FIGSIZE = (10, 6)
BAR_WIDTH = 0.85
BAR_ALPHA = 0.80
ERRORBAR_COLOR = "#202020"
ERRORBAR_LINEWIDTH = 1.0
ERRORBAR_CAPSIZE = 5

POINT_JITTER = 0.165
POINT_SIZE = 31
POINT_ALPHA = 0.88
POINT_EDGE_COLOR = "black"
POINT_EDGE_WIDTH = 0.45
POINT_RANDOM_SEED = 123

DRAW_SEED_LINES = False  # useful for paired seed designs, but can be visually busy.
ROTATE_XTICKS = 0
GRID_ALPHA = 0.28
DPI = 300
SAVE_PNG = True
SAVE_SVG = True
SHOW_FIGURES = True

# Significance annotation style.
# Compact letters are placed above bars.
# Bars that share at least one letter are not significantly different
# based on Holm-adjusted pairwise tests.
SHOW_COMPACT_LETTERS = True
LETTER_FONT_SIZE = 10.5
# LETTER_FONT_WEIGHT = "bold"
LETTER_Y_OFFSET_FRACTION = 0.045
SHOW_OMNIBUS_P_IN_TITLE = True
SHOW_SIGNIFICANCE_NOTE = True

# Per-bar star-significance style.
# Each selected strategy is compared against REFERENCE_STRATEGY_FOR_TEST.
# The adjusted star sign is placed above that bar; S1 itself is marked as "ref".
SHOW_PAIRWISE_STAR_ANNOTATIONS = True
STAR_ANNOTATE_NS = True
STAR_FONT_SIZE = 10.0
# STAR_FONT_WEIGHT = "bold"
STAR_COLOR = "#222222"
STAR_TEXT_OFFSET_FRACTION = 0.045
STAR_Y_EXTRA_FRACTION = 0.16

SHOW_BAR_VALUE_LABELS = True
BAR_VALUE_FORMAT = ".5g"
BAR_VALUE_LABEL_SIZE = 10
BAR_VALUE_LABEL_COLOR = "#222222"
BAR_VALUE_LABEL_OFFSET_FRACTION = 0.025

# Output folder name. Only figure files are saved; no sub CSV files are written.
OUTPUT_FOLDER_NAME = "selected_control_seed_barplots"

# Metrics requested by the user.
CONTROL_METRICS = {
    # "control_pseudo_local_mixing_score": {
    #     "label": "Local mixing score",
    #     "ylabel": "Opposite-neighbor fraction",
    #     "direction": "higher",
    #     "save_name": "local_mixing_score_barplot",
    # },
    "mmd_pca": {
        "label": "MMD in PCA space",
        "ylabel": "MMD distance",
        "direction": "lower",
        "save_name": "mmd_pca_barplot",
    },
}

# Source-column fallback if the canonical metric columns are absent.
METRIC_SOURCE_FALLBACKS = {
    # "control_pseudo_local_mixing_score": [
    #     "control_pseudo_local_mixing_score",
    #     "source_mixing_opposite_neighbor_fraction_mean",
    #     "local_mixing_score",
    # ],
    "mmd_pca": [
        "mmd_pca",
        "mmd_rbf_pca",
        "mmd_pca_mean",
    ],
}

In [ ]:
# ============================================================
# Strategy labels and colors
# ============================================================

# STRATEGY_PLOT_LABELS = {
#     "S0_naive_mean_control_reference": "Naive\nmean\ncontrol",
#     "S1_random_single_control": "Random\nsingle\ncontrol",
#     "S2_random_average_controls": "Random\naverage\ncontrol",
#     "S3_SEACell_metacell_average": "Random\nmetacell\naverage",
#     "S4_SEACell_balanced_random_sample": "SEACell\nbalanced\nsample",
#     "S5_SEACell_OT_sampled_average": "OT sampled\naverage",
# }
STRATEGY_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive\nmean\ncontrol",
    "S1_random_single_control": "Random\nsingle\ncontrol",
    "S2_random_average_controls": "Random\naverage\ncontrol",
    "S4_SEACell_balanced_random_sample": "Metacell\nbalanced\nrandom",
    "S3_SEACell_metacell_average": "Random\nmetacell\naverage",
    "S5_SEACell_OT_sampled_average": "Metacell OT\nsampled\naverage",
}

STRATEGY_BASE_COLORS = {
    "S0_naive_mean_control_reference": "#D0E0EF",
    "S1_random_single_control": "#6E8FB2",
    "S2_random_average_controls": "#7DA494",
    "S3_SEACell_metacell_average": "#E5A79A",
    "S4_SEACell_balanced_random_sample": "#EAB67A",
    "S5_SEACell_OT_sampled_average": "#9F8DB8",
}

# Same variant-aware palette used in the scatter plots.
S5_VARIANT_COLORS = {
    "200&5": "#D49AB5",
    "350&5": "#9F8DB8",
    "500&5": "#B66699",
}

DEFAULT_STRATEGY_RENAME_MAP = {
    "S0_naive_mean_control_reference": "S0_naive_mean_control_reference",
    "S1_random_single_control": "S1_random_single_control",
    "S2_random_average_controls": "S2_random_average_controls",
    "S3_SEACell_metacell_average": "S3_SEACell_metacell_average",
    "S4_SEACell_balanced_random_sample": "S4_SEACell_balanced_random_sample",
    "S5_SEACell_OT_sampled_average": "S5_SEACell_OT_sampled_average",
    "S0": "S0_naive_mean_control_reference",
    "S1": "S1_random_single_control",
    "S2": "S2_random_average_controls",
    "S3": "S3_SEACell_metacell_average",
    "S4": "S4_SEACell_balanced_random_sample",
    "S5": "S5_SEACell_OT_sampled_average",
    "S4_random_single_control_oracle": "S1_random_single_control",
    "S4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control_cell": "S1_random_single_control",
    "S3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_control_cells": "S2_random_average_controls",
    "S5_random_metacell_average": "S3_SEACell_metacell_average",
    "S3_random_metacell_average": "S3_SEACell_metacell_average",
    "strategy5_random_metacell_average": "S3_SEACell_metacell_average",
    "S1_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "S4_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "strategy1_seacell_balanced_random_repeated": "S4_SEACell_balanced_random_sample",
    "S2_SEACell_OT_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "S2_SEACell_OT_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
    "strategy2_seacells_ot_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "strategy2_seacell_ot_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
}

STRATEGY_ORDER_MAP = {
    "S0_naive_mean_control_reference": 0,
    "S1_random_single_control": 1,
    "S2_random_average_controls": 2,
    "S3_SEACell_metacell_average": 3,
    "S4_SEACell_balanced_random_sample": 4,
    "S5_SEACell_OT_sampled_average": 5,
}

In [ ]:
# ============================================================
# Basic IO helpers
# ============================================================

def read_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table format: {path}")


def as_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def is_missing(x: Any) -> bool:
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def is_valid_color(x: Any) -> bool:
    if is_missing(x):
        return False
    x = str(x).strip()
    return bool(x) and x.lower() not in {"nan", "none", "null"}


def fmt_int_like(x: Any) -> str:
    if is_missing(x):
        return "NA"
    try:
        return str(int(round(float(x))))
    except Exception:
        return str(x)


def clean_label(label: str) -> str:
    return " ".join(str(label).replace("\n", " ").split())


def extract_variant_suffix(display_label: str) -> str:
    display_label = str(display_label)
    if "(" in display_label and ")" in display_label:
        return display_label[display_label.rfind("("):].strip()
    return ""


def extract_number_from_text(x: Any, patterns: Iterable[str]) -> float:
    text = str(x)
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            try:
                return float(m.group(1))
            except Exception:
                pass
    return np.nan


def get_dataset_group_title(path: Path) -> str:
    dataset = path.parent.name.replace("_pseudo_pairing_evaluation", "")
    group = path.name
    return f"{dataset} | {group}"


# ============================================================
# Variant harmonization and selected-table matching
# ============================================================

def infer_strategy_column(df: pd.DataFrame) -> str:
    for col in ["strategy", "strategy_id", "pairing_strategy"]:
        if col in df.columns:
            return col
    raise KeyError(f"Cannot infer strategy column from columns: {list(df.columns)}")


def fill_numeric_from_candidates(df: pd.DataFrame, target: str, candidates: list[str]) -> None:
    if target not in df.columns:
        df[target] = np.nan
    df[target] = pd.to_numeric(df[target], errors="coerce")
    for col in candidates:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            df[target] = df[target].where(df[target].notna(), vals)


def fill_from_text_patterns(df: pd.DataFrame, target: str, text_cols: list[str], patterns: list[str]) -> None:
    if target not in df.columns:
        df[target] = np.nan
    df[target] = pd.to_numeric(df[target], errors="coerce")
    for col in text_cols:
        if col not in df.columns:
            continue
        vals = df[col].map(lambda x: extract_number_from_text(x, patterns))
        df[target] = df[target].where(df[target].notna(), vals)


def make_variant_label(row: pd.Series | dict[str, Any]) -> str:
    strategy = str(row.get("strategy", ""))
    nmc = row.get("n_metacells", np.nan)
    topk = row.get("top_k", np.nan)
    sampled = row.get("sampled_metacells_k", np.nan)

    if strategy in {
        "S0_naive_mean_control_reference",
        "S1_random_single_control",
        "S2_random_average_controls",
    }:
        return "default"

    if strategy == "S3_SEACell_metacell_average":
        parts = []
        if not is_missing(nmc):
            parts.append(f"nmc_{fmt_int_like(nmc)}")
        if not is_missing(sampled):
            parts.append(f"sampledMC_{fmt_int_like(sampled)}")
        return "__".join(parts) if parts else "default"

    if strategy == "S4_SEACell_balanced_random_sample":
        return f"nmc_{fmt_int_like(nmc)}" if not is_missing(nmc) else "default"

    if strategy == "S5_SEACell_OT_sampled_average":
        parts = []
        if not is_missing(nmc):
            parts.append(f"nmc_{fmt_int_like(nmc)}")
        if not is_missing(topk):
            parts.append(f"topk_{fmt_int_like(topk)}")
        return "__".join(parts) if parts else "default"

    return "default"


def make_display_variant_label(row: pd.Series | dict[str, Any]) -> str:
    strategy = str(row.get("strategy", ""))
    nmc = row.get("n_metacells", np.nan)
    topk = row.get("top_k", np.nan)
    sampled = row.get("sampled_metacells_k", np.nan)

    if strategy == "S3_SEACell_metacell_average" and not is_missing(nmc) and not is_missing(sampled):
        return f"{strategy} ({fmt_int_like(nmc)}&{fmt_int_like(sampled)})"
    if strategy == "S4_SEACell_balanced_random_sample" and not is_missing(nmc):
        return f"{strategy} ({fmt_int_like(nmc)})"
    if strategy == "S5_SEACell_OT_sampled_average" and not is_missing(nmc) and not is_missing(topk):
        return f"{strategy} ({fmt_int_like(nmc)}&{fmt_int_like(topk)})"
    return strategy


def make_variant_id(row: pd.Series | dict[str, Any]) -> str:
    strategy = str(row.get("strategy", ""))
    label = make_variant_label(row)
    return strategy if label == "default" else f"{strategy}__{label}"


def canonicalize_variants(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    strategy_col = infer_strategy_column(out)
    out["strategy_old"] = out[strategy_col].astype(str)
    out["strategy"] = out["strategy_old"].map(DEFAULT_STRATEGY_RENAME_MAP).fillna(out["strategy_old"])
    out["strategy_order"] = out["strategy"].map(STRATEGY_ORDER_MAP).fillna(99).astype(int)

    fill_numeric_from_candidates(out, "n_metacells", ["n_metacells_requested", "n_metacells_observed"])
    fill_numeric_from_candidates(out, "n_metacells_requested", ["n_metacells"])
    fill_numeric_from_candidates(out, "n_metacells_observed", ["n_metacells"])
    fill_numeric_from_candidates(out, "top_k", ["top_k_metacells"])
    fill_numeric_from_candidates(out, "top_k_metacells", ["top_k"])
    fill_numeric_from_candidates(out, "sampled_metacells_k", ["n_metacells_to_average"])
    fill_numeric_from_candidates(out, "n_metacells_to_average", ["sampled_metacells_k"])
    fill_numeric_from_candidates(out, "n_control_cells_to_average", [])
    fill_numeric_from_candidates(out, "sample_cells_per_metacell", [])

    text_cols = [
        c for c in ["parameter_label", "seacell_setting_id", "run_id", "strategy_old", "display_variant_label", "variant_id"]
        if c in out.columns
    ]
    fill_from_text_patterns(out, "n_metacells", text_cols, [r"nmc[_=-]?(\d+)", r"metacell[s]?[_=-]?(\d+)"])
    fill_from_text_patterns(out, "n_metacells_requested", text_cols, [r"nmc[_=-]?(\d+)"])
    fill_from_text_patterns(out, "top_k", text_cols, [r"topk[_=-]?(\d+)", r"top_k[_=-]?(\d+)"])
    fill_from_text_patterns(out, "top_k_metacells", text_cols, [r"topk[_=-]?(\d+)", r"top_k[_=-]?(\d+)"])
    fill_from_text_patterns(
        out,
        "sampled_metacells_k",
        text_cols,
        [r"sampledMC[_=-]?(\d+)", r"sampled_metacells[_=-]?(\d+)", r"(?:^|__)k[_=-]?(\d+)"],
    )
    fill_from_text_patterns(
        out,
        "n_metacells_to_average",
        text_cols,
        [r"sampledMC[_=-]?(\d+)", r"sampled_metacells[_=-]?(\d+)", r"(?:^|__)k[_=-]?(\d+)"],
    )

    out["variant_label"] = out.apply(make_variant_label, axis=1)
    out["display_variant_label"] = out.apply(make_display_variant_label, axis=1)
    out["variant_id"] = out.apply(make_variant_id, axis=1)
    return out


def add_standard_control_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for metric, sources in METRIC_SOURCE_FALLBACKS.items():
        if metric not in out.columns:
            out[metric] = np.nan
        out[metric] = pd.to_numeric(out[metric], errors="coerce")
        for src in sources:
            if src in out.columns:
                vals = pd.to_numeric(out[src], errors="coerce")
                out[metric] = out[metric].where(out[metric].notna(), vals)
    return out


def infer_seed_column(df: pd.DataFrame) -> str:
    for col in ["sampling_seed", "seed", "pair_selection_seed", "random_seed", "run_seed"]:
        if col in df.columns:
            return col
    df["_pseudo_seed_index"] = df.groupby("variant_id").cumcount()
    return "_pseudo_seed_index"


def get_final_label(row: pd.Series) -> str:
    for col in ["final_strategy_label", "plot_label"]:
        if col in row.index and not is_missing(row[col]) and str(row[col]).strip():
            return clean_label(str(row[col]))
    if "display_variant_label" in row.index and not is_missing(row["display_variant_label"]):
        display = str(row["display_variant_label"])
        suffix = extract_variant_suffix(display)
        base = STRATEGY_PLOT_LABELS.get(str(row.get("strategy", "")), str(row.get("strategy", "")))
        return clean_label(f"{base} {suffix}" if suffix else base)
    return clean_label(str(row.get("variant_id", row.get("strategy", "variant"))))


def choose_color(row: pd.Series) -> str:
    for col in ["manual_color", "color", "plot_color"]:
        if col in row.index and is_valid_color(row[col]):
            return str(row[col]).strip()

    strategy = str(row.get("strategy", ""))
    display = str(row.get("display_variant_label", ""))
    if strategy == "S5_SEACell_OT_sampled_average":
        for key, color in S5_VARIANT_COLORS.items():
            if key in display:
                return color
    return STRATEGY_BASE_COLORS.get(strategy, "#999999")


def load_selected_variants(selection_path: str | Path) -> pd.DataFrame:
    selected = read_table(selection_path)
    selected = canonicalize_variants(selected)
    if "select_for_final" in selected.columns:
        selected = selected[as_bool_series(selected["select_for_final"])].copy()
    if not INCLUDE_S0_AS_BAR:
        selected = selected[selected["strategy"] != "S0_naive_mean_control_reference"].copy()
    if selected.empty:
        raise RuntimeError(f"No selected variants found in {selection_path}")

    selected["final_label"] = selected.apply(get_final_label, axis=1)
    selected["plot_color"] = selected.apply(choose_color, axis=1)

    selected = selected.sort_values(
        ["strategy_order", "n_metacells", "top_k", "sampled_metacells_k", "variant_id"],
        na_position="first",
    ).drop_duplicates("variant_id", keep="first")
    return selected.reset_index(drop=True)


def find_control_seed_table(sub_path: Path) -> Path:
    candidates = [
        sub_path / "result_analysis" / "aggregated_by_task" / "control_manifold" / "control_manifold_canonical_input_with_required_metrics.csv",
        sub_path / "control_manifold" / "control_manifold_preservation_repeated_run_summary.csv",
        sub_path / "control_manifold" / "control_manifold_preservation_repeated_summary.csv",
        sub_path / "control_manifold" / "control_manifold_preservation_repeated_long.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Cannot find control-manifold seed-level table. Tried:\n"
        + "\n".join(str(p) for p in candidates)
    )


def load_control_seed_metrics(sub_path: Path, selected: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    control_path = find_control_seed_table(sub_path)
    raw = read_table(control_path)
    raw = canonicalize_variants(raw)
    raw = add_standard_control_metrics(raw)
    seed_col = infer_seed_column(raw)

    metric_cols = list(CONTROL_METRICS.keys())
    keep_cols = [
        "variant_id", "strategy", "strategy_order", "display_variant_label",
        "n_metacells", "top_k", "sampled_metacells_k", seed_col,
    ] + [m for m in metric_cols if m in raw.columns]
    keep_cols = [c for c in keep_cols if c in raw.columns]
    raw = raw[keep_cols].copy()
    raw = raw.rename(columns={seed_col: "sampling_seed_for_plot"})

    # Keep selected variants for bars, keep S0 as a dashed reference line,
    # and keep S1/random-single as the statistical reference even if it is not selected as a bar.
    selected_ids = selected["variant_id"].astype(str).tolist()
    wanted_ids = selected_ids.copy()
    for extra_id in [NAIVE_BASELINE_ID, REFERENCE_STRATEGY_FOR_TEST]:
        if extra_id not in wanted_ids:
            wanted_ids.append(extra_id)
    seed_df = raw[raw["variant_id"].astype(str).isin(wanted_ids)].copy()

    missing_ids = sorted(set(selected_ids) - set(seed_df["variant_id"].astype(str)))
    missing = selected[selected["variant_id"].isin(missing_ids)].copy()

    seed_df = seed_df.merge(
        selected[["variant_id", "final_label", "plot_color"]],
        on="variant_id",
        how="left",
    )

    # Fill S0 reference metadata, because S0 is usually not part of the selected bar table.
    s0_mask = seed_df["variant_id"].astype(str) == NAIVE_BASELINE_ID
    if s0_mask.any():
        seed_df.loc[s0_mask, "final_label"] = STRATEGY_PLOT_LABELS.get(NAIVE_BASELINE_ID, "Naive average control")
        seed_df.loc[s0_mask, "plot_color"] = STRATEGY_BASE_COLORS.get(NAIVE_BASELINE_ID, "#BDBDBD")

    # If a table contains multiple rows per variant/seed, reduce to one value per seed.
    group_cols = ["variant_id", "sampling_seed_for_plot"]
    meta_cols = ["strategy", "strategy_order", "display_variant_label", "final_label", "plot_color"]
    agg = {m: "mean" for m in metric_cols if m in seed_df.columns}
    for c in meta_cols:
        if c in seed_df.columns:
            agg[c] = "first"
    seed_df = seed_df.groupby(group_cols, dropna=False, as_index=False).agg(agg)

    return seed_df, missing

In [ ]:
# ============================================================
# Statistical testing
# ============================================================

def infer_paired_design(seed_df: pd.DataFrame, metric: str, variant_order: list[str]) -> bool:
    sets = []
    for vid in variant_order:
        vals = seed_df[(seed_df["variant_id"] == vid) & seed_df[metric].notna()]
        sets.append(set(vals["sampling_seed_for_plot"].tolist()))
    if not sets:
        return False
    first = sets[0]
    return len(first) >= 2 and all(s == first for s in sets)


def holm_adjust(p_values: list[float]) -> list[float]:
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    if m == 0:
        return []
    order = np.argsort(p)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = (m - rank) * p[idx]
        adj = max(adj, running_max)
        adjusted[idx] = min(adj, 1.0)
        running_max = adjusted[idx]
    return adjusted.tolist()


def p_to_stars(p: float) -> str:
    if pd.isna(p):
        return "ns"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def compute_statistics(
    seed_df: pd.DataFrame,
    metric: str,
    variant_order: list[str],
    stat_mode: str,
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    data = seed_df[["variant_id", "sampling_seed_for_plot", metric]].dropna().copy()

    if stat_mode not in {"auto", "paired", "unpaired"}:
        raise ValueError("STAT_MODE must be 'auto', 'paired', or 'unpaired'.")

    paired = infer_paired_design(data, metric, variant_order) if stat_mode == "auto" else (stat_mode == "paired")
    mode_used = "paired" if paired else "unpaired"

    # Omnibus test
    omnibus_records = []
    if len(variant_order) >= 3:
        if paired:
            pivot = data.pivot_table(index="sampling_seed_for_plot", columns="variant_id", values=metric, aggfunc="mean")
            complete = pivot[variant_order].dropna()
            if complete.shape[0] >= 2:
                stat, p = stats.friedmanchisquare(*[complete[v].values for v in variant_order])
                omnibus_records.append({
                    "metric": metric,
                    "test": "Friedman test",
                    "paired": True,
                    "n_groups": len(variant_order),
                    "n_complete_seed_blocks": int(complete.shape[0]),
                    "statistic": float(stat),
                    "p_value": float(p),
                })
            else:
                omnibus_records.append({
                    "metric": metric,
                    "test": "Friedman test",
                    "paired": True,
                    "n_groups": len(variant_order),
                    "n_complete_seed_blocks": int(complete.shape[0]),
                    "statistic": np.nan,
                    "p_value": np.nan,
                    "note": "Too few complete seed blocks for Friedman test.",
                })
        else:
            arrays = [data.loc[data["variant_id"] == v, metric].values for v in variant_order]
            arrays = [a for a in arrays if len(a) >= 2]
            if len(arrays) >= 3:
                stat, p = stats.kruskal(*arrays, nan_policy="omit")
                omnibus_records.append({
                    "metric": metric,
                    "test": "Kruskal-Wallis test",
                    "paired": False,
                    "n_groups": len(arrays),
                    "statistic": float(stat),
                    "p_value": float(p),
                })
            else:
                omnibus_records.append({
                    "metric": metric,
                    "test": "Kruskal-Wallis test",
                    "paired": False,
                    "n_groups": len(arrays),
                    "statistic": np.nan,
                    "p_value": np.nan,
                    "note": "Too few groups with n>=2 for Kruskal-Wallis test.",
                })
    elif len(variant_order) == 2:
        v1, v2 = variant_order
        if paired:
            pivot = data.pivot_table(index="sampling_seed_for_plot", columns="variant_id", values=metric, aggfunc="mean")
            pair = pivot[[v1, v2]].dropna()
            if pair.shape[0] >= 2:
                diff = pair[v1].values - pair[v2].values
                if np.allclose(diff, 0):
                    stat, p = 0.0, 1.0
                else:
                    stat, p = stats.wilcoxon(pair[v1].values, pair[v2].values)
                omnibus_records.append({"metric": metric, "test": "Wilcoxon signed-rank test", "paired": True, "statistic": float(stat), "p_value": float(p)})
        else:
            a = data.loc[data["variant_id"] == v1, metric].values
            b = data.loc[data["variant_id"] == v2, metric].values
            stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
            omnibus_records.append({"metric": metric, "test": "Mann-Whitney U test", "paired": False, "statistic": float(stat), "p_value": float(p)})

    # Pairwise tests with Holm correction.
    pair_records = []
    if paired:
        pivot = data.pivot_table(index="sampling_seed_for_plot", columns="variant_id", values=metric, aggfunc="mean")

    for i in range(len(variant_order) - 1):
        for j in range(i + 1, len(variant_order)):
            v1, v2 = variant_order[i], variant_order[j]
            if paired:
                pair = pivot[[v1, v2]].dropna()
                n = int(pair.shape[0])
                if n < 2:
                    stat, p = np.nan, np.nan
                    test = "Wilcoxon signed-rank test"
                else:
                    diff = pair[v1].values - pair[v2].values
                    if np.allclose(diff, 0):
                        stat, p = 0.0, 1.0
                    else:
                        stat, p = stats.wilcoxon(pair[v1].values, pair[v2].values)
                    test = "Wilcoxon signed-rank test"
            else:
                a = data.loc[data["variant_id"] == v1, metric].values
                b = data.loc[data["variant_id"] == v2, metric].values
                n = int(min(len(a), len(b)))
                if len(a) < 2 or len(b) < 2:
                    stat, p = np.nan, np.nan
                else:
                    stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
                test = "Mann-Whitney U test"

            pair_records.append({
                "metric": metric,
                "group1": v1,
                "group2": v2,
                "group1_index": i,
                "group2_index": j,
                "test": test,
                "paired": paired,
                "n_used": n,
                "statistic": float(stat) if np.isfinite(stat) else np.nan,
                "p_value": float(p) if np.isfinite(p) else np.nan,
            })

    pairwise = pd.DataFrame(pair_records)
    if not pairwise.empty:
        valid_mask = pairwise["p_value"].notna()
        adjusted = [np.nan] * len(pairwise)
        adj_valid = holm_adjust(pairwise.loc[valid_mask, "p_value"].tolist())
        for idx, adj in zip(pairwise.index[valid_mask], adj_valid):
            adjusted[idx] = adj
        pairwise["p_adj_holm"] = adjusted
        pairwise["significance"] = pairwise["p_adj_holm"].map(p_to_stars)
        pairwise["significant"] = pairwise["p_adj_holm"] < ALPHA

    omnibus = pd.DataFrame(omnibus_records)
    return omnibus, pairwise, mode_used

def compute_reference_pairwise_statistics(
    seed_df: pd.DataFrame,
    metric: str,
    variant_order: list[str],
    reference_variant: str = REFERENCE_STRATEGY_FOR_TEST,
    stat_mode: str = STAT_MODE,
) -> tuple[pd.DataFrame, str]:
    """Compare every selected strategy against one reference strategy.

    Main behavior:
        - S0 is excluded from selected bars and statistical tests.
        - Each selected strategy is compared against S1_random_single_control.
        - If seed IDs are shared, use paired t-test.
        - If seed IDs are not shared, use Welch's unpaired t-test.
        - Holm correction is applied across all reference-vs-strategy comparisons.

    This is more appropriate for the current visualization because the bars show
    mean ± SD across seeds, so the statistical test should test the paired mean
    difference rather than rank signs only.
    """
    data = seed_df[["variant_id", "sampling_seed_for_plot", metric]].dropna().copy()
    data["variant_id"] = data["variant_id"].astype(str)

    if reference_variant not in set(data["variant_id"]):
        raise RuntimeError(
            f"Cannot run reference significance tests because {reference_variant!r} "
            f"is absent from the seed-level data for metric {metric!r}."
        )

    if stat_mode not in {"auto", "paired", "unpaired"}:
        raise ValueError("STAT_MODE must be 'auto', 'paired', or 'unpaired'.")

    records: list[dict[str, Any]] = []

    reference_values = data.loc[data["variant_id"] == reference_variant].copy()

    ref_by_seed = reference_values.pivot_table(
        index="sampling_seed_for_plot",
        values=metric,
        aggfunc="mean",
    )

    for i, vid in enumerate(variant_order):
        vid = str(vid)

        # ------------------------------------------------------------
        # Reference bar itself
        # ------------------------------------------------------------
        if vid == reference_variant:
            records.append(
                {
                    "metric": metric,
                    "reference_group": reference_variant,
                    "group2": vid,
                    "group2_index": i,
                    "test": "reference",
                    "paired": np.nan,
                    "n_used": int(reference_values[metric].notna().sum()),
                    "mean_reference": float(reference_values[metric].mean()),
                    "mean_group2": float(reference_values[metric].mean()),
                    "mean_difference_group2_minus_reference": 0.0,
                    "statistic": np.nan,
                    "p_value": np.nan,
                    "p_adj_holm": np.nan,
                    "significance": REFERENCE_BAR_TEXT,
                    "significant": False,
                }
            )
            continue

        candidate = data.loc[data["variant_id"] == vid].copy()

        if candidate.empty:
            records.append(
                {
                    "metric": metric,
                    "reference_group": reference_variant,
                    "group2": vid,
                    "group2_index": i,
                    "test": "missing candidate data",
                    "paired": False,
                    "n_used": 0,
                    "mean_reference": float(reference_values[metric].mean()),
                    "mean_group2": np.nan,
                    "mean_difference_group2_minus_reference": np.nan,
                    "statistic": np.nan,
                    "p_value": np.nan,
                }
            )
            continue

        cand_by_seed = candidate.pivot_table(
            index="sampling_seed_for_plot",
            values=metric,
            aggfunc="mean",
        )

        paired_frame = ref_by_seed.join(
            cand_by_seed,
            how="inner",
            lsuffix="_ref",
            rsuffix="_cand",
        ).dropna()

        if stat_mode == "paired":
            paired_used = True
        elif stat_mode == "unpaired":
            paired_used = False
        else:
            paired_used = paired_frame.shape[0] >= 2

        # ------------------------------------------------------------
        # Paired t-test across matched seed IDs
        # ------------------------------------------------------------
        if paired_used:
            n = int(paired_frame.shape[0])
            test = "paired t-test vs random single control"

            if n < 2:
                stat = p = np.nan
                mean_ref = float(paired_frame[f"{metric}_ref"].mean()) if n > 0 else np.nan
                mean_cand = float(paired_frame[f"{metric}_cand"].mean()) if n > 0 else np.nan
                mean_diff = mean_cand - mean_ref if n > 0 else np.nan
            else:
                a = paired_frame[f"{metric}_ref"].to_numpy(dtype=float)
                b = paired_frame[f"{metric}_cand"].to_numpy(dtype=float)

                # Candidate minus reference. Positive means candidate is higher.
                diff = b - a

                mean_ref = float(np.mean(a))
                mean_cand = float(np.mean(b))
                mean_diff = float(np.mean(diff))

                if np.allclose(diff, 0):
                    stat, p = 0.0, 1.0
                else:
                    # If all paired differences are almost identical but nonzero,
                    # scipy ttest_rel can return inf/nan due to near-zero variance.
                    # In this deterministic seed-summary setting, this should be
                    # treated as an extremely consistent difference.
                    diff_sd = float(np.std(diff, ddof=1)) if n > 1 else np.nan

                    if np.isfinite(diff_sd) and np.isclose(diff_sd, 0.0) and not np.isclose(mean_diff, 0.0):
                        stat = np.inf if mean_diff > 0 else -np.inf
                        p = 0.0
                    else:
                        stat, p = stats.ttest_rel(
                            b,
                            a,
                            nan_policy="omit",
                            alternative="two-sided",
                        )

        # ------------------------------------------------------------
        # Fallback: Welch's t-test when seed IDs are not matched
        # ------------------------------------------------------------
        else:
            a = reference_values[metric].dropna().to_numpy(dtype=float)
            b = candidate[metric].dropna().to_numpy(dtype=float)

            n = int(min(len(a), len(b)))
            test = "Welch t-test vs random single control"

            mean_ref = float(np.mean(a)) if len(a) > 0 else np.nan
            mean_cand = float(np.mean(b)) if len(b) > 0 else np.nan
            mean_diff = mean_cand - mean_ref if np.isfinite(mean_ref) and np.isfinite(mean_cand) else np.nan

            if len(a) < 2 or len(b) < 2:
                stat = p = np.nan
            elif np.allclose(a.mean(), b.mean()) and np.allclose(a.std(ddof=1), 0) and np.allclose(b.std(ddof=1), 0):
                stat, p = 0.0, 1.0
            else:
                stat, p = stats.ttest_ind(
                    b,
                    a,
                    equal_var=False,
                    nan_policy="omit",
                    alternative="two-sided",
                )

        records.append(
            {
                "metric": metric,
                "reference_group": reference_variant,
                "group2": vid,
                "group2_index": i,
                "test": test,
                "paired": paired_used,
                "n_used": n,
                "mean_reference": mean_ref,
                "mean_group2": mean_cand,
                "mean_difference_group2_minus_reference": mean_diff,
                "statistic": float(stat) if not pd.isna(stat) else np.nan,
                "p_value": float(p) if not pd.isna(p) else np.nan,
            }
        )

    pairwise = pd.DataFrame(records)

    if not pairwise.empty:
        valid_mask = pairwise["p_value"].notna()

        adjusted = [np.nan] * len(pairwise)
        adj_valid = holm_adjust(pairwise.loc[valid_mask, "p_value"].tolist())

        for idx, adj in zip(pairwise.index[valid_mask], adj_valid):
            adjusted[idx] = adj

        pairwise["p_adj_holm"] = adjusted
        pairwise["significance"] = pairwise["p_adj_holm"].map(p_to_stars)

        pairwise.loc[
            pairwise["group2"].astype(str) == reference_variant,
            "significance",
        ] = REFERENCE_BAR_TEXT

        pairwise["significant"] = pairwise["p_adj_holm"] < ALPHA

    mode_used = (
        f"each selected strategy vs {REFERENCE_STRATEGY_FOR_TEST_LABEL}; "
        f"paired t-test for matched seeds, Welch t-test otherwise; Holm correction"
    )

    return pairwise, mode_used


def _letters_for_indices(n: int) -> list[str]:
    base = list("abcdefghijklmnopqrstuvwxyz")
    if n <= len(base):
        return base[:n]
    letters = base.copy()
    for first in base:
        for second in base:
            letters.append(first + second)
            if len(letters) >= n:
                return letters
    return [f"L{i + 1}" for i in range(n)]


def compact_letter_display(
    pairwise: pd.DataFrame,
    variant_order: list[str],
    alpha: float = ALPHA,
) -> dict[str, str]:
    """Return compact-letter labels from Holm-adjusted pairwise tests.

    Bars that share at least one letter are not significantly different.
    Bars that do not share any letter are significantly different.

    The implementation enumerates maximal cliques in the graph of non-significant
    pairwise relations. This handles non-transitive cases, e.g. A~B, B~C, A!=C,
    by assigning A=a, B=ab, C=b.
    """
    n = len(variant_order)
    if n == 0:
        return {}
    if n == 1 or pairwise.empty:
        return {v: "a" for v in variant_order}

    idx = {v: i for i, v in enumerate(variant_order)}
    sig = np.zeros((n, n), dtype=bool)

    for _, row in pairwise.iterrows():
        g1 = str(row.get("group1", ""))
        g2 = str(row.get("group2", ""))
        if g1 not in idx or g2 not in idx:
            continue
        p_adj = row.get("p_adj_holm", np.nan)
        is_sig = bool(pd.notna(p_adj) and float(p_adj) < alpha)
        i, j = idx[g1], idx[g2]
        sig[i, j] = sig[j, i] = is_sig

    def is_nonsig_clique(subset: tuple[int, ...]) -> bool:
        for i, j in combinations(subset, 2):
            if sig[i, j]:
                return False
        return True

    all_cliques: list[frozenset[int]] = []
    for size in range(1, n + 1):
        for subset in combinations(range(n), size):
            if is_nonsig_clique(subset):
                all_cliques.append(frozenset(subset))

    maximal_cliques = []
    for c in all_cliques:
        if not any(c < other for other in all_cliques):
            maximal_cliques.append(c)

    maximal_cliques = sorted(maximal_cliques, key=lambda c: (min(c), -len(c), sorted(c)))
    letters = _letters_for_indices(len(maximal_cliques))

    out = {v: "" for v in variant_order}
    for letter, clique in zip(letters, maximal_cliques):
        for i in sorted(clique):
            out[variant_order[i]] += letter

    for v in variant_order:
        if not out[v]:
            out[v] = "a"
    return out


In [ ]:
# ============================================================
# Plotting
# ============================================================

def metric_summary_for_order(metric_df: pd.DataFrame, metric: str, variant_order: list[str]) -> tuple[list[np.ndarray], np.ndarray, np.ndarray, np.ndarray]:
    values = []
    means = []
    stds = []
    ns = []
    for vid in variant_order:
        vals = pd.to_numeric(metric_df.loc[metric_df["variant_id"] == vid, metric], errors="coerce").dropna().to_numpy(dtype=float)
        values.append(vals)
        means.append(float(np.nanmean(vals)) if len(vals) else np.nan)
        stds.append(float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0)
        ns.append(int(len(vals)))
    return values, np.asarray(means), np.asarray(stds), np.asarray(ns)


def make_bar_xtick_label(row: pd.Series, n: int) -> str:
    """Build bar x-axis label from STRATEGY_PLOT_LABELS, optionally with variant suffix."""
    strategy = str(row.get("strategy", ""))

    if USE_STRATEGY_PLOT_LABELS_FOR_XTICKS:
        base = STRATEGY_PLOT_LABELS.get(strategy, clean_label(str(row.get("final_label", strategy))))
    else:
        base = clean_label(str(row.get("final_label", strategy)))

    suffix = ""
    if ADD_VARIANT_SUFFIX_TO_XTICKS:
        suffix = extract_variant_suffix(str(row.get("display_variant_label", "")))

    label = base if not suffix else f"{base}\n{suffix}"
    return f"{label}\nn={int(n)}"


def plot_metric_barplot(
    seed_df: pd.DataFrame,
    selected: pd.DataFrame,
    metric: str,
    outdir: str | Path,
    dataset_group_title: str,
) -> dict[str, Any]:
    info = CONTROL_METRICS[metric]
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    available_ids = set(seed_df["variant_id"].astype(str))
    variant_order = [v for v in selected["variant_id"].astype(str).tolist() if v in available_ids]
    if len(variant_order) < 2:
        raise RuntimeError(f"Need at least two selected variants with data for {metric}.")

    selected_lookup = selected.set_index("variant_id")
    colors = [selected_lookup.loc[v, "plot_color"] for v in variant_order]

    metric_df = seed_df[["variant_id", "sampling_seed_for_plot", metric]].dropna().copy()
    metric_df = metric_df[
        metric_df["variant_id"].astype(str).isin(set(variant_order) | {REFERENCE_STRATEGY_FOR_TEST})
    ].copy()

    # Bar summaries use selected variants only; S1 can still be retained in metric_df
    # solely as the statistical reference if it was not selected as a bar.
    bar_metric_df = metric_df[metric_df["variant_id"].astype(str).isin(variant_order)].copy()
    y_values, means, stds, ns = metric_summary_for_order(bar_metric_df, metric, variant_order)

    baseline_values = pd.to_numeric(
        seed_df.loc[seed_df["variant_id"].astype(str) == NAIVE_BASELINE_ID, metric],
        errors="coerce",
    ).dropna().to_numpy(dtype=float)
    baseline_mean = float(np.nanmean(baseline_values)) if len(baseline_values) else np.nan
    baseline_std = float(np.nanstd(baseline_values, ddof=1)) if len(baseline_values) > 1 else np.nan

    # Significance tests: each selected strategy is compared against random single control.
    # S0 remains a dashed reference line and is not included in these tests.
    omnibus = pd.DataFrame()
    pairwise, mode_used = compute_reference_pairwise_statistics(
        seed_df=metric_df,
        metric=metric,
        variant_order=variant_order,
        reference_variant=REFERENCE_STRATEGY_FOR_TEST,
        stat_mode=STAT_MODE,
    )

    # Add readable labels inside returned statistics object, but do not save CSV files.
    if not pairwise.empty:
        label_map = dict(zip(selected["variant_id"], selected["final_label"]))
        pairwise = pairwise.copy()
        pairwise["reference_label"] = pairwise["reference_group"].map(label_map).fillna(REFERENCE_STRATEGY_FOR_TEST_LABEL)
        pairwise["group2_label"] = pairwise["group2"].map(label_map)

    fig, ax = plt.subplots(figsize=FIGSIZE)
    positions = np.arange(len(variant_order), dtype=float)

    ax.bar(
        positions,
        means,
        yerr=stds,
        width=BAR_WIDTH,
        color=colors,
        edgecolor="black",
        linewidth=0.9,
        alpha=BAR_ALPHA,
        capsize=ERRORBAR_CAPSIZE,
        error_kw={
            "ecolor": ERRORBAR_COLOR,
            "elinewidth": ERRORBAR_LINEWIDTH,
            "capthick": ERRORBAR_LINEWIDTH,
        },
        zorder=2,
    )

    rng = np.random.default_rng(POINT_RANDOM_SEED)
    seed_to_jitter: dict[Any, float] = {}

    for pos, vals, color, vid in zip(positions, y_values, colors, variant_order):
        sub = metric_df.loc[metric_df["variant_id"] == vid, ["sampling_seed_for_plot", metric]].dropna().copy()
        xs = []
        for seed in sub["sampling_seed_for_plot"].tolist():
            if DRAW_SEED_LINES:
                # Keep the same horizontal offset for the same seed across bars.
                if seed not in seed_to_jitter:
                    seed_to_jitter[seed] = float(rng.uniform(-POINT_JITTER, POINT_JITTER))
                xs.append(pos + seed_to_jitter[seed])
            else:
                xs.append(pos + float(rng.uniform(-POINT_JITTER, POINT_JITTER)))

        ax.scatter(
            xs,
            sub[metric].to_numpy(dtype=float),
            s=POINT_SIZE,
            color=color,
            edgecolor=POINT_EDGE_COLOR,
            linewidth=POINT_EDGE_WIDTH,
            alpha=POINT_ALPHA,
            zorder=4,
        )

    if DRAW_SEED_LINES:
        pivot = metric_df.pivot_table(index="sampling_seed_for_plot", columns="variant_id", values=metric, aggfunc="mean")
        for _, row in pivot[variant_order].dropna(how="all").iterrows():
            ax.plot(positions, row.values, color="#999999", linewidth=0.6, alpha=0.35, zorder=1)

    # ------------------------------------------------------------------
    # Pairwise star-significance annotations.
    # Each bracket corresponds to one Holm-adjusted pairwise comparison.
    # ------------------------------------------------------------------
    star_pairs = pd.DataFrame()

    finite_upper = means + np.nan_to_num(stds, nan=0.0)
    all_raw_values = [np.asarray(v, dtype=float) for v in y_values if len(v) > 0]
    if len(baseline_values):
        all_raw_values.append(np.asarray(baseline_values, dtype=float))

    data_min = float(np.nanmin([np.nanmin(v) for v in all_raw_values])) if all_raw_values else 0.0
    data_max = float(np.nanmax([np.nanmax(v) for v in all_raw_values])) if all_raw_values else float(np.nanmax(finite_upper))
    bar_max = float(np.nanmax(finite_upper))
    baseline_top = baseline_mean if np.isfinite(baseline_mean) else data_max
    local_mixing_half_top = (
        LOCAL_MIXING_HALF_REFERENCE_VALUE
        if metric == "control_pseudo_local_mixing_score" and SHOW_LOCAL_MIXING_HALF_REFERENCE
        else data_max
    )
    # base_top = max(data_max, bar_max, baseline_top, local_mixing_half_top)
    # span = max(float(base_top - data_min), abs(base_top) * 0.08, 1e-8)
    base_top = max(data_max, bar_max, baseline_top, local_mixing_half_top)
    span = max(float(base_top - data_min), abs(base_top) * 0.08, 1e-8)

    # ------------------------------------------------------------------
    # Exact bar-value labels.
    # These show the mean value used as the bar height.
    # ------------------------------------------------------------------
    bar_value_y_lookup: dict[int, float] = {}

    if SHOW_BAR_VALUE_LABELS:
        for i, (pos, mean_value, std_value, vals) in enumerate(
            zip(positions, means, stds, y_values)
        ):
            if not np.isfinite(mean_value):
                continue

            y_bar_top = mean_value + (std_value if np.isfinite(std_value) else 0.0)
            y_data_top = np.nanmax(vals) if len(vals) else y_bar_top

            y_text = max(y_bar_top, y_data_top) + BAR_VALUE_LABEL_OFFSET_FRACTION * span
            bar_value_y_lookup[i] = y_text

            ax.text(
                pos,
                y_text,
                f"{mean_value:{BAR_VALUE_FORMAT}}",
                ha="center",
                va="bottom",
                fontsize=BAR_VALUE_LABEL_SIZE,
                color=BAR_VALUE_LABEL_COLOR,
                clip_on=False,
                zorder=8,
            )

    # S0 naive average control baseline as a horizontal dashed line.
    if SHOW_NAIVE_BASELINE_LINE and np.isfinite(baseline_mean):
        ax.axhline(
            baseline_mean,
            color=NAIVE_BASELINE_LINE_COLOR,
            linestyle=NAIVE_BASELINE_LINESTYLE,
            linewidth=NAIVE_BASELINE_LINEWIDTH,
            alpha=NAIVE_BASELINE_LINE_ALPHA,
            zorder=10,
        )
        x_text = positions[-1] + NAIVE_BASELINE_TEXT_X_FRACTION * BAR_WIDTH
        y_text = baseline_mean + NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION * span
        line_label = f"{NAIVE_BASELINE_TEXT}: {baseline_mean:.4g}"
        if np.isfinite(baseline_std):
            line_label += f" ± {baseline_std:.2g}"
        ax.text(
            x_text,
            y_text,
            line_label,
            ha="right",
            va="bottom",
            fontsize=NAIVE_BASELINE_TEXT_SIZE,
            color=NAIVE_BASELINE_TEXT_COLOR,
            clip_on=False,
            zorder=10,
        )

    # Local mixing score reference at 0.5.
    if metric == "control_pseudo_local_mixing_score" and SHOW_LOCAL_MIXING_HALF_REFERENCE:
        ax.axhline(
            LOCAL_MIXING_HALF_REFERENCE_VALUE,
            color=LOCAL_MIXING_HALF_REFERENCE_COLOR,
            linestyle=LOCAL_MIXING_HALF_REFERENCE_LINESTYLE,
            linewidth=LOCAL_MIXING_HALF_REFERENCE_LINEWIDTH,
            alpha=LOCAL_MIXING_HALF_REFERENCE_ALPHA,
            zorder=10.4,
        )
        ax.text(
            positions[2] + 0.45 * BAR_WIDTH,
            LOCAL_MIXING_HALF_REFERENCE_VALUE + 0.012 * span,
            LOCAL_MIXING_HALF_REFERENCE_TEXT,
            ha="right",
            va="bottom",
            fontsize=LOCAL_MIXING_HALF_REFERENCE_TEXT_SIZE,
            color=LOCAL_MIXING_HALF_REFERENCE_TEXT_COLOR,
            clip_on=False,
            zorder=10,
        )

    if SHOW_PAIRWISE_STAR_ANNOTATIONS and not pairwise.empty:
        star_pairs = pairwise.copy()

        # Place one star/ns label above each bar. Each p-value compares that bar
        # against random single control only. S1 itself is marked as reference.
        star_lookup = {
            str(row["group2"]): str(row["significance"])
            for _, row in star_pairs.iterrows()
        }

        text_offset = STAR_TEXT_OFFSET_FRACTION * span
        y_star_values = []

        for i, vid in enumerate(variant_order):
            label = star_lookup.get(vid, "ns")
            if vid == REFERENCE_STRATEGY_FOR_TEST and not SHOW_REFERENCE_BAR_TEXT:
                continue
            if label == "ns" and not STAR_ANNOTATE_NS:
                continue

            y_bar_top = means[i] + (stds[i] if np.isfinite(stds[i]) else 0.0)
            y_data_top = np.nanmax(y_values[i]) if len(y_values[i]) else y_bar_top
            # y_text = max(y_bar_top, y_data_top) + 0.1 * text_offset
            y_text = max(y_bar_top, y_data_top, bar_value_y_lookup.get(i, -np.inf),) + 0.65 * text_offset
            y_star_values.append(y_text)

            ax.text(
                positions[i],
                y_text,
                label,
                ha="center",
                va="bottom",
                fontsize=STAR_FONT_SIZE,
                # fontweight=STAR_FONT_WEIGHT,
                color=STAR_COLOR,
                clip_on=False,
                zorder=8,
            )

        # top_needed = max(base_top, max(y_star_values) if y_star_values else base_top) + STAR_Y_EXTRA_FRACTION * span
        bar_value_tops = list(bar_value_y_lookup.values())
        top_needed = max(base_top, max(y_star_values) if y_star_values else base_top, max(bar_value_tops) if bar_value_tops else base_top,) + STAR_Y_EXTRA_FRACTION * span
        ax.set_ylim(top=top_needed)
    else:
        ax.set_ylim(top=base_top + 0.18 * span)

    # Keep lower bound slightly below zero for positive metrics, otherwise pad the data range.
    current_ylim = ax.get_ylim()
    if data_min >= 0:
        ax.set_ylim(bottom=-0.02, top=current_ylim[1])
    else:
        ax.set_ylim(bottom=data_min - 0.08 * span, top=current_ylim[1])

    ax.set_title(
        f"{info['label']} across selected variants\n{dataset_group_title}",
        # f"{info['label']} across selected variants\n{dataset_group_title} | statistics: {mode_used}",
        fontsize=18, weight="bold")
    # ax.text(0.5, 1.02, dataset_group_title, transform=ax.transAxes, ha="center", va="bottom", fontsize=13, color="#555555",)

    ax.set_ylabel(info["ylabel"], fontsize=15)
    ax.set_xticks(positions)
    xtick_labels = [make_bar_xtick_label(selected_lookup.loc[v], n) for v, n in zip(variant_order, ns)]
    ax.set_xticklabels(xtick_labels, rotation=ROTATE_XTICKS, ha="center", fontsize=13)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=GRID_ALPHA)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if SHOW_SIGNIFICANCE_NOTE and SHOW_PAIRWISE_STAR_ANNOTATIONS:
        note = f"Stars/ns: Holm-adjusted tests vs {REFERENCE_STRATEGY_FOR_TEST_LABEL}; S0 shown only as baseline."
        ax.text(
            0.01,
            0.98,
            note,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=10,
            color="#555555",
        )

    fig.tight_layout()

    fig_base = outdir / info["save_name"]
    if SAVE_PNG:
        png_path = fig_base.with_suffix(".png")
        fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
        print(f"[Saved] {png_path}")
    if SAVE_SVG:
        svg_path = fig_base.with_suffix(".svg")
        fig.savefig(svg_path, bbox_inches="tight")
        print(f"[Saved] {svg_path}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)

    return {
        "metric": metric,
        "figure_base": fig_base,
        "stat_mode_used": mode_used,
        "omnibus": omnibus,
        "pairwise": pairwise,
        "star_pairs": star_pairs,
        "means": dict(zip(variant_order, means)),
        "stds": dict(zip(variant_order, stds)),
        "ns": dict(zip(variant_order, ns)),
        "naive_baseline_mean": baseline_mean,
        "naive_baseline_std": baseline_std,
        "naive_baseline_n": int(len(baseline_values)),
    }

def process_one_sub_path(sub_path: Path) -> dict[str, Any]:
    print(f"\n[Processing] {sub_path}")
    selection_path = sub_path / "result_analysis" / "selected_variants_TEMPLATE_EDIT_ME.csv"
    if not selection_path.exists():
        print(f"[Skip] Missing selection table: {selection_path}")
        return {"sub_path": sub_path, "skipped": True, "reason": "missing selection table"}

    selected = load_selected_variants(selection_path)
    seed_df, missing = load_control_seed_metrics(sub_path, selected)

    outdir = sub_path / "result_analysis" / OUTPUT_FOLDER_NAME
    outdir.mkdir(parents=True, exist_ok=True)

    if not missing.empty:
        missing_labels = ", ".join(missing["final_label"].astype(str).tolist())
        print(f"[Warning] Selected variants missing in control seed table: {missing_labels}")

    outputs = []
    title = get_dataset_group_title(sub_path)
    for metric in CONTROL_METRICS:
        if metric not in seed_df.columns or seed_df[metric].notna().sum() == 0:
            print(f"[Skip] No values for metric: {metric}")
            continue
        outputs.append(plot_metric_barplot(seed_df, selected, metric, outdir, title))

    return {"sub_path": sub_path, "outdir": outdir, "outputs": outputs}


# ============================================================
# Main execution
# ============================================================

def main() -> None:
    candidate_paths = [root_dir / f"{name}_pseudo_pairing_evaluation" for name in dataset_names]
    detailed_sub_paths = [sub_path / group for sub_path in candidate_paths for group in groups]
    valid_paths = [sub_path for sub_path in detailed_sub_paths if sub_path.exists()]

    if RUN_ALL_VALID_PATHS:
        targets = valid_paths
    else:
        if SELECTED_SUB_PATH is None:
            if not valid_paths:
                raise RuntimeError("No valid dataset/group paths found.")
            targets = [valid_paths[-1]]
        else:
            targets = [Path(SELECTED_SUB_PATH)]

    if not targets:
        raise RuntimeError("No dataset/group paths selected for plotting.")

    for target in targets:
        try:
            process_one_sub_path(Path(target))
        except Exception as e:
            print(f"[Error] Failed on {target}: {e}")
            raise


if __name__ == "__main__":
    main()